# Protein Subcellular Localization with ESM-2 — high-accuracy recipe

Fine-tune Meta AI's **ESM-2 (650M)** with a **learnable attention-pooling head** to predict
where a protein localizes in the cell, on the **DeepLoc** benchmark.

This notebook pulls the accuracy levers that matter on this benchmark:
1. **ESM-2 650M** encoder (vs. 150M).
2. **Attention pooling** over residues instead of the CLS token (localization signals are
   local — a learnable per-residue weight recovers them; cf. Light Attention, Stärk 2021).
3. **Full 1024-residue context with dual-end truncation** — keeps both termini, where sorting
   signals live (signal peptides N-terminal; PTS1 / KDEL C-terminal).
4. **Accuracy-optimized loss** — plain cross-entropy (class weighting trades accuracy for macro-F1).
5. **Cosine schedule, 4 epochs.**

**Runtime:** Colab **T4 GPU** (`Runtime → Change runtime type → T4 GPU`). ~1.5–2.5 h.
> Honest note: on this 10-class benchmark, published single models land ~78–83% accuracy.
> This recipe targets the low-to-mid 80s; 85%+ is at the top of the field and not guaranteed.

## 1 · Setup

In [ ]:
!pip -q install -U transformers datasets umap-learn scikit-learn seaborn accelerate

In [ ]:
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), 'Enable the T4 GPU: Runtime -> Change runtime type -> T4 GPU'
device = 'cuda'
print(torch.cuda.get_device_name(0))

## 2 · Load the DeepLoc benchmark

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

raw = load_dataset('proteinea/deeploc')
train_full = raw['train'].to_pandas()[['input','loc']].rename(columns={'input':'sequence','loc':'label'})
test = raw['test'].to_pandas()[['input','loc']].rename(columns={'input':'sequence','loc':'label'})
train_full = train_full.dropna().drop_duplicates('sequence')
test = test.dropna().drop_duplicates('sequence')
train, val = train_test_split(train_full, test_size=0.1, stratify=train_full['label'], random_state=42)
print(f'{len(train)} train / {len(val)} val / {len(test)} test')
train['label'].value_counts()

## 3 · Tokenize (full context, dual-end truncation)

For proteins longer than the window we keep the **first and last** residues rather than
head-truncating, so the C-terminal signal survives.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'   # 650M. Use esm2_t30_150M_UR50D for a fast ~2x-quicker run.
MAX_LENGTH = 1024

labels = sorted(train['label'].unique())
label2id = {l:i for i,l in enumerate(labels)}; id2label = {i:l for l,i in label2id.items()}
num_labels = len(labels)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def dual_end(s, n):
    if len(s) <= n: return s
    h = n // 2; return s[:h] + s[-(n-h):]

def to_ds(df):
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    def tok(b):
        seqs = [dual_end(s, MAX_LENGTH-2) for s in b['sequence']]
        t = tokenizer(seqs, truncation=True, max_length=MAX_LENGTH)
        t['labels'] = [label2id[l] for l in b['label']]
        return t
    return ds.map(tok, batched=True, remove_columns=['sequence','label'])

train_ds, val_ds, test_ds = to_ds(train), to_ds(val), to_ds(test)
collator = DataCollatorWithPadding(tokenizer)

## 4 · Model: ESM-2 encoder + attention pooling

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class EsmAttentionClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.esm = AutoModel.from_pretrained(model_name)
        h = self.esm.config.hidden_size
        self.attn = nn.Linear(h, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(nn.Linear(2*h, h), nn.GELU(), nn.Dropout(dropout), nn.Linear(h, num_labels))
        self.config = self.esm.config; self.config.num_labels = num_labels
    def gradient_checkpointing_enable(self, **kw):
        self.esm.gradient_checkpointing_enable(**kw)
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        H = self.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pad = (attention_mask == 0).unsqueeze(-1)
        w = torch.softmax(self.attn(H).masked_fill(pad, float('-inf')), dim=1)
        attn_pool = (w * H).sum(1)
        max_pool = H.masked_fill(pad, float('-inf')).max(1).values
        logits = self.classifier(self.dropout(torch.cat([attn_pool, max_pool], -1)))
        loss = None if labels is None else nn.functional.cross_entropy(logits, labels)
        return {'loss': loss, 'logits': logits}

model = EsmAttentionClassifier(MODEL_NAME, num_labels)
model.gradient_checkpointing_enable()   # fit 650M on a 16GB T4
model.esm.config.use_cache = False

## 5 · Fine-tune

If you hit CUDA OOM, drop `per_device_train_batch_size` to 1 (and raise
`gradient_accumulation_steps` to 32) or lower `MAX_LENGTH` to 768.

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

def compute_metrics(p):
    pred = np.argmax(p.predictions, axis=-1)
    return {'accuracy': accuracy_score(p.label_ids, pred),
            'macro_f1': f1_score(p.label_ids, pred, average='macro'),
            'mcc': matthews_corrcoef(p.label_ids, pred)}

args = TrainingArguments(
    output_dir='outputs', num_train_epochs=4,
    per_device_train_batch_size=2, per_device_eval_batch_size=4,
    gradient_accumulation_steps=16, learning_rate=2e-5,
    lr_scheduler_type='cosine', warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='accuracy', greater_is_better=True, save_total_limit=1,
    fp16=True, logging_steps=20, report_to='none')

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics, data_collator=collator)
trainer.train()

## 6 · Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style='whitegrid')

test_metrics = trainer.evaluate(test_ds)
print({k: round(v,4) for k,v in test_metrics.items() if isinstance(v,float)})

pred = trainer.predict(test_ds)
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(9,8))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, values_format='.2f', cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix (row-normalized) — Test'); plt.tight_layout(); plt.show()

## 7 · Embedding map (from the fine-tuned encoder)

In [ ]:
import umap
@torch.no_grad()
def embed(seqs, bs=8):
    model.eval(); vecs=[]
    for i in range(0,len(seqs),bs):
        s=[dual_end(x, MAX_LENGTH-2) for x in seqs[i:i+bs]]
        b=tokenizer(s, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LENGTH).to(device)
        h=model.esm(**b).last_hidden_state; m=b['attention_mask'].unsqueeze(-1).float()
        vecs.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).float().cpu().numpy())
    return np.concatenate(vecs)

emb = embed(test['sequence'].tolist())
xy = umap.UMAP(n_neighbors=25, min_dist=0.3, metric='cosine', random_state=42).fit_transform(emb)
plt.figure(figsize=(11,8.5))
for c,color in zip(labels, sns.color_palette('tab10', len(labels))):
    m = test['label'].values==c; plt.scatter(xy[m,0], xy[m,1], s=10, alpha=0.6, color=color, label=c, linewidths=0)
plt.legend(markerscale=2, fontsize=9); plt.title('Fine-tuned ESM-2 embeddings (UMAP) by subcellular location')
plt.tight_layout(); plt.show()

## 8 · Refresh the live dashboard

Regenerate `results.js` from this model and download it. Drop it into `docs/` in the repo
and push — GitHub Pages updates the live dashboard within a minute.

In [ ]:
import json
from sklearn.metrics import precision_recall_fscore_support

logits = pred.predictions
probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
conf = probs.max(1)
met = {'accuracy': float(accuracy_score(y_true,y_pred)), 'macro_f1': float(f1_score(y_true,y_pred,average='macro')),
       'weighted_f1': float(f1_score(y_true,y_pred,average='weighted')), 'mcc': float(matthews_corrcoef(y_true,y_pred)),
       'n_test': int(len(y_true))}
ids = list(range(len(labels)))
pr,rc,f1c,sup = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
per_class = [{'label':labels[i],'precision':float(pr[i]),'recall':float(rc[i]),'f1':float(f1c[i]),'support':int(sup[i])} for i in ids]
cm_c = confusion_matrix(y_true, y_pred, labels=ids); cm_n = cm_c / cm_c.sum(1, keepdims=True).clip(min=1)
tr = train['label'].value_counts().to_dict()
distribution = [{'label':c,'count':int(tr.get(c,0))} for c in labels]
points = [{'x':round(float(xy[i,0]),3),'y':round(float(xy[i,1]),3),'t':int(y_true[i]),'p':int(probs[i].argmax()),'c':round(float(conf[i]),3)} for i in range(len(y_true))]
seqs = test['sequence'].tolist(); examples=[]
for ci,c in enumerate(labels):
    mask=(y_true==ci)&(y_pred==ci)
    if not mask.any(): continue
    idxs=np.where(mask)[0]; best=idxs[conf[idxs].argmax()]; order=probs[best].argsort()[::-1][:3]
    examples.append({'true':c,'length':len(seqs[best]),'preview':seqs[best][:60],
        'top3':[{'label':labels[j],'prob':round(float(probs[best,j]),3)} for j in order]})
results = {'model': MODEL_NAME, 'source': 'fine-tuned ESM-2 650M + attention pooling',
  'dataset': {'name':'DeepLoc (proteinea/deeploc)','n_train':int(len(train)),'n_test':int(len(y_true)),'n_classes':len(labels)},
  'classes': labels, 'metrics': met, 'per_class': per_class,
  'confusion': {'counts': cm_c.astype(int).tolist(), 'normalized': np.round(cm_n,3).tolist()},
  'distribution': distribution, 'points': points, 'examples': examples}
with open('results.js','w') as fh:
    fh.write('// Auto-generated by the fine-tuning notebook.\n')
    fh.write('window.RESULTS = ' + json.dumps(results, separators=(',',':')) + ';\n')
print('Test accuracy %.3f | macro-F1 %.3f  ->  results.js written' % (met['accuracy'], met['macro_f1']))
from google.colab import files; files.download('results.js')

---
Model: [ESM-2](https://huggingface.co/facebook/esm2_t33_650M_UR50D) (Lin et al., 2023) ·
Data: [DeepLoc](https://services.healthtech.dtu.dk/services/DeepLoc-2.0/) (Almagro Armenteros et al., 2017) ·
Pooling: Light Attention (Stärk et al., 2021)